# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad-ahmed-developer/flyRank_Internship_Tasks/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

ML-04 already selected the five features for the Refresh / Content Opportunity Scoring lane. I am not selecting new features here.

The approved feature vector is:

1. `imp_prev30`
2. `clicks_prev30`
3. `avg_position_prev30`
4. `content_age_days`
5. `days_since_last_update`

The first three are constructed from the 30 days before the March 31 decision point.

The final two describe the content state at the decision point.

Identifiers are retained separately for grouping and auditing but are not included in `X`.

No future outcome information is used to construct the final feature vector.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb huggingface_hub

In [2]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Token order:
# environment variable -> Colab Secret -> prompt as last resort
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# Exact file/directory structure from the working warehouse notebook.
TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected to FlyRank warehouse.")
print("Development decision point: 2026-03-31")

Connected to FlyRank warehouse.
Development decision point: 2026-03-31


In [4]:
daily_schema = con.sql(
    f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}"
).df()

content_schema = con.sql(
    f"DESCRIBE SELECT * FROM {TABLES['dim_content']}"
).df()

daily_columns = daily_schema["column_name"].tolist()
content_columns = content_schema["column_name"].tolist()

print("Daily fact columns:")
print(daily_columns)

print("\nContent dimension columns:")
print(content_columns)

Daily fact columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

Content dimension columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optim

In [5]:
required_daily_columns = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]

missing_daily_columns = [
    col for col in required_daily_columns
    if col not in daily_columns
]

if missing_daily_columns:
    raise ValueError(
        f"Required daily columns are missing: {missing_daily_columns}"
    )

print("All required daily feature columns are present.")

All required daily feature columns are present.


In [6]:
CONTENT_CREATED = "content_created_date"
CONTENT_UPDATED = "content_updated_date"

print("Content creation column:", CONTENT_CREATED)
print("Content update column:", CONTENT_UPDATED)

Content creation column: content_created_date
Content update column: content_updated_date


In [7]:
DECISION_DATE = "2026-03-31"

date_check = con.sql(
    f"""
    SELECT
        COUNT(*) AS total_rows,

        SUM(
            CASE
                WHEN CAST(content_created_date AS DATE)
                     > DATE '{DECISION_DATE}'
                THEN 1
                ELSE 0
            END
        ) AS created_after_decision,

        SUM(
            CASE
                WHEN CAST(content_updated_date AS DATE)
                     > DATE '{DECISION_DATE}'
                THEN 1
                ELSE 0
            END
        ) AS updated_after_decision,

        MIN(CAST(content_created_date AS DATE)) AS earliest_created,
        MAX(CAST(content_created_date AS DATE)) AS latest_created,

        MIN(CAST(content_updated_date AS DATE)) AS earliest_updated,
        MAX(CAST(content_updated_date AS DATE)) AS latest_updated

    FROM {TABLES['dim_content']}
    """
).df()

date_check

,total_rows,created_after_decision,updated_after_decision,earliest_created,latest_created,earliest_updated,latest_updated
0,519606,86172.0,382739.0,2024-10-16,2026-07-06,2024-10-28,2026-07-06


In [8]:
content_state = con.sql(
    f"""
    SELECT
        content_hash_id,

        DATE_DIFF(
            'day',
            CAST(content_created_date AS DATE),
            DATE '{DECISION_DATE}'
        ) AS content_age_days

    FROM {TABLES['dim_content']}

    WHERE CAST(content_created_date AS DATE)
          <= DATE '{DECISION_DATE}'
    """
).df()

print(f"Content rows existing by decision date: {len(content_state):,}")

content_state.head()

Content rows existing by decision date: 433,434


,content_hash_id,content_age_days
0,content_004e9c4c32e88631,175
1,content_0236ef736698e17c,175
2,content_025f6cfd3c298870,175
3,content_0263d5f9b7a2ecd4,175
4,content_02752c6c1c60161f,175


In [9]:
content_update_state = con.sql(
    f"""
    SELECT
        content_hash_id,

        CASE
            WHEN CAST(content_updated_date AS DATE)
                 <= DATE '{DECISION_DATE}'
            THEN DATE_DIFF(
                'day',
                CAST(content_updated_date AS DATE),
                DATE '{DECISION_DATE}'
            )
            ELSE NULL
        END AS days_since_last_update

    FROM {TABLES['dim_content']}

    WHERE CAST(content_created_date AS DATE)
          <= DATE '{DECISION_DATE}'
    """
).df()

content_update_state.head()

,content_hash_id,days_since_last_update
0,content_004e9c4c32e88631,<NA>
1,content_0236ef736698e17c,<NA>
2,content_025f6cfd3c298870,<NA>
3,content_0263d5f9b7a2ecd4,<NA>
4,content_02752c6c1c60161f,<NA>


In [10]:
content_state = content_state.merge(
    content_update_state,
    on="content_hash_id",
    how="left"
)

print("Content-state shape:", content_state.shape)

content_state.head()

Content-state shape: (433434, 3)


,content_hash_id,content_age_days,days_since_last_update
0,content_004e9c4c32e88631,175,<NA>
1,content_0236ef736698e17c,175,<NA>
2,content_025f6cfd3c298870,175,<NA>
3,content_0263d5f9b7a2ecd4,175,<NA>
4,content_02752c6c1c60161f,175,<NA>


In [11]:
print("Missing values:")
print(
    content_state[
        ["content_age_days", "days_since_last_update"]
    ].isna().sum()
)

print("\nNegative values:")
print(
    (
        content_state[
            ["content_age_days", "days_since_last_update"]
        ] < 0
    ).sum()
)

Missing values:
content_age_days               0
days_since_last_update    296567
dtype: int64

Negative values:
content_age_days          0
days_since_last_update    0
dtype: Int64


In [12]:
# Build the historical 30-day features for the March 31, 2026
# decision point.

historical_features = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_prev30,

        SUM(gsc_clicks) AS clicks_prev30,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN
                SUM(gsc_sum_position)
                / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_prev30

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date <= DATE '2026-03-30'

      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

print(f"Historical feature rows: {len(historical_features):,}")

print("\nHistorical feature columns:")
print(historical_features.columns.tolist())

historical_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Historical feature rows: 175,205

Historical feature columns:
['client_hash_id', 'content_hash_id', 'imp_prev30', 'clicks_prev30', 'avg_position_prev30']


,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1105.0,2.0,4.447059
1,client_73cda7b4e4f265ea,content_05597932fe4da067,54.0,0.0,1.925926
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,148.0,0.0,5.675676
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1369.0,6.0,6.990504
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2682.0,16.0,3.871738


In [13]:
missing_update = (
    content_state["days_since_last_update"].isna().sum()
)

total_content = len(content_state)

print(
    f"Missing days_since_last_update: "
    f"{missing_update:,} / {total_content:,}"
)

print(
    "Missing percentage:",
    round(missing_update / total_content * 100, 2),
    "%"
)

Missing days_since_last_update: 296,567 / 433,434
Missing percentage: 68.42 %


In [14]:
FEATURES = [
    "imp_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "content_age_days",
    "days_since_last_update",
]

feature_frame = historical_features.merge(
    content_state,
    on="content_hash_id",
    how="left"
)

feature_frame = feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        *FEATURES
    ]
].copy()

print("Feature frame shape:", feature_frame.shape)

print("\nFeature columns:")
print(FEATURES)

feature_frame.head()

Feature frame shape: (175205, 7)

Feature columns:
['imp_prev30', 'clicks_prev30', 'avg_position_prev30', 'content_age_days', 'days_since_last_update']


,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30,content_age_days,days_since_last_update
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1105.0,2.0,4.447059,396,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,54.0,0.0,1.925926,396,<NA>
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,148.0,0.0,5.675676,396,<NA>
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1369.0,6.0,6.990504,396,<NA>
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2682.0,16.0,3.871738,396,<NA>


In [15]:
duplicate_decision_rows = (
    feature_frame
    .groupby(["client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
)

print(
    "Duplicate client/content feature rows:",
    len(duplicate_decision_rows)
)

assert len(duplicate_decision_rows) == 0

print("Feature-frame grain check passed.")

Duplicate client/content feature rows: 0
Feature-frame grain check passed.


In [16]:
comparison = con.sql(
    f"""
    SELECT
        COUNT(*) AS all_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS available_true_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS FALSE
        ) AS available_false_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS NULL
        ) AS availability_null_rows

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date <= DATE '2026-03-30'
    """
).df()

comparison

,all_rows,available_true_rows,available_false_rows,availability_null_rows
0,9509942,3485839,6024103,0


In [17]:
print("Current historical feature rows:", len(historical_features))

print(
    "Rows excluded by gsc_data_available filter:",
    331436 - len(historical_features)
)

Current historical feature rows: 175205
Rows excluded by gsc_data_available filter: 156231


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The approved feature vector contains five numeric features. No categorical feature is included, so no categorical encoding is required in this notebook.

### 1. `imp_prev30` — previous-30-day impressions

This is the number of Google Search Console impressions observed for the content item during the previous 30-day window before the March 31, 2026 decision point.

- **Meaning:** historical search visibility.
- **Missing handling:** no additional fill is applied here; the feature was constructed from the available historical performance rows.
- **Available when?** Yes. It belongs to the historical window before the decision point, so it is available before the future outcome is evaluated.

### 2. `clicks_prev30` — previous-30-day clicks

This is the number of Google Search Console clicks observed during the previous 30-day window.

- **Meaning:** historical search traffic generated from Google Search.
- **Missing handling:** no additional fill is applied here; the feature comes from the historical performance aggregation.
- **Available when?** Yes. It comes from the previous 30-day window and is therefore known before the prediction moment.

### 3. `avg_position_prev30` — previous-30-day average position

This is the average Google Search Console search position calculated over the previous 30-day window.

- **Meaning:** historical search ranking performance.
- **Missing handling:** no additional fill is applied here; the value is retained from the historical aggregation.
- **Available when?** Yes. It describes performance before the decision point and does not use the future outcome window.

### 4. `content_age_days` — content age at the decision point

This is the number of days between `content_created_date` and March 31, 2026.

- **Meaning:** how old the content was when the decision would have been made.
- **Missing handling:** none in the constructed content-state table. The March 31 eligibility filter produced 0 missing values.
- **Available when?** Yes, because the creation date is already known for content that existed by the decision point.

### 5. `days_since_last_update` — time since the recorded update date

This is the number of days between `content_updated_date` and March 31, 2026, when the recorded update date is on or before the decision point.

- **Meaning:** how recently the content had been updated at the decision point.
- **Missing handling:** 296,567 of 433,434 eligible content rows (68.42%) are missing. These values are not replaced with zero because zero would incorrectly mean that the content was updated on the decision date. The missingness reflects that the current warehouse snapshot does not provide a valid pre-March-31 update state for those rows.
- **Available when?** Only when the recorded update date is on or before the decision point. Future update dates are not used as historical information.

### Categorical handling

No categorical feature is part of the approved five-feature vector, so no categorical encoding is performed.

### Overall availability rule

The feature vector is intended to contain only information that could be known at the March 31, 2026 decision point. Future outcome information is not used as a feature. IDs remain context for joining and identifying rows and are not included in the model feature vector.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# summarize availability and missingness of the five approved features.

feature_availability = pd.DataFrame({
    "feature": FEATURES,
    "missing_rows": [
        feature_frame[f].isna().sum()
        for f in FEATURES
    ]
})

feature_availability["total_rows"] = len(feature_frame)

feature_availability["missing_pct"] = (
    feature_availability["missing_rows"]
    / feature_availability["total_rows"]
    * 100
)

feature_availability

,feature,missing_rows,total_rows,missing_pct
0,imp_prev30,0,175205,0.000000
1,clicks_prev30,0,175205,0.000000
2,avg_position_prev30,0,175205,0.000000
3,content_age_days,0,175205,0.000000
4,days_since_last_update,147325,175205,84.087212


In [19]:
print("Categorical features:")
print("None — all five approved features are numeric.")

print("\nNegative values:")
print(
    (
        feature_frame[FEATURES] < 0
    ).sum()
)

Categorical features:
None — all five approved features are numeric.

Negative values:
imp_prev30                0
clicks_prev30             0
avg_position_prev30       0
content_age_days          0
days_since_last_update    0
dtype: Int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The five approved features are intended to be known at the March 31, 2026 decision point. I will now deliberately attack this feature vector for leakage.

The main risks are:

1. using information from the future outcome window as a feature;
2. using a column derived directly from the label;
3. using a product or workflow flag that would only be known after the decision.

For this test, the future 30-day outcome will be kept separate from the feature vector. I will use the April 2026 outcome only to create an observed decline label. I will then deliberately create an unsafe feature from that future information and compare it with the honest feature vector.

The leaked feature is only for demonstration. It will be removed before the final feature vector is kept.

A feature is considered safe only if its value could have been known at the March 31 decision moment. The future outcome can be used to evaluate a prediction, but it must not be supplied to the model as an input.

In [20]:
# Future outcome window:
# April 1, 2026 through April 30, 2026.
#
# This information is NOT part of the feature vector.
# It is used only to construct the observed future outcome.

FUTURE_START = "2026-04-01"
FUTURE_END = "2026-04-30"

future_outcome = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_next30,

        COUNT(DISTINCT report_date) AS available_future_days

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '{FUTURE_START}'
      AND report_date <= DATE '{FUTURE_END}'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

print("Future outcome rows:", len(future_outcome))

print("\nFuture outcome columns:")
print(future_outcome.columns.tolist())

print("\nAvailable future-day distribution:")
print(
    future_outcome["available_future_days"]
    .describe()
)

future_outcome.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Future outcome rows: 194760

Future outcome columns:
['client_hash_id', 'content_hash_id', 'imp_next30', 'available_future_days']

Available future-day distribution:
count    194760.000000
mean         20.030088
std          10.985802
min           1.000000
25%           9.000000
50%          25.000000
75%          30.000000
max          30.000000
Name: available_future_days, dtype: float64


,client_hash_id,content_hash_id,imp_next30,available_future_days
0,client_62f4a7e64f5e0096,content_76c1f31e2b38f054,249.0,28
1,client_62f4a7e64f5e0096,content_ffc5ab4b34aab1f8,281.0,30
2,client_62f4a7e64f5e0096,content_9739856fc83dc1ca,722.0,30
3,client_62f4a7e64f5e0096,content_3d1dc691a3502105,3068.0,30
4,client_62f4a7e64f5e0096,content_9d28af4f99c5e67b,3896.0,30


### Future outcome availability

The future outcome is measured from April 1–30, 2026.

The previous feature window contains 30 days, so the future outcome should also represent a complete 30-day window. The warehouse contains uneven GSC availability, so I will not treat missing future days as zero impressions.

For the leakage experiment, I therefore restrict the observed outcome to client-content pairs with all 30 future days available. This avoids labeling a page as declining merely because fewer future days were observed.

The future outcome remains evaluation information only; it is not included in the approved feature vector.

In [21]:
# Check how many future observations have the complete 30-day window.

future_coverage = (
    future_outcome["available_future_days"]
    .value_counts()
    .sort_index()
    .rename_axis("available_future_days")
    .reset_index(name="client_content_pairs")
)

print("Future-window coverage:")
display(future_coverage)

complete_future = future_outcome[
    future_outcome["available_future_days"] == 30
].copy()

print(
    "\nClient-content pairs with complete future window:",
    len(complete_future)
)

print(
    "Percentage of future-outcome rows with 30 days:",
    round(
        len(complete_future) / len(future_outcome) * 100,
        2
    ),
    "%"
)

Future-window coverage:


,available_future_days,client_content_pairs
0,1,12680
1,2,7934
2,3,5908
3,4,4673
4,5,4137
5,6,4128
6,7,3572
7,8,3659
8,9,3285
9,10,3205



Client-content pairs with complete future window: 74572
Percentage of future-outcome rows with 30 days: 38.29 %


### Construct the leakage-audit dataset

I now join the observed April outcome to the March 31 decision-time feature frame using the client-content identifiers.

The five approved features remain unchanged. The April impression total is attached only so that I can construct the observed outcome and test leakage.

The join is performed on both `client_hash_id` and `content_hash_id` because the same content identifier can exist across clients. The identifiers are used only for joining and auditing, not as model features.

In [23]:
# Keep only the complete 30-day future outcome.
complete_future = complete_future[
    ["client_hash_id", "content_hash_id", "imp_next30"]
].copy()

# Join future outcome to the decision-time feature frame.
leakage_audit = feature_frame.merge(
    complete_future,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Feature-frame rows:", len(feature_frame))
print("Complete future rows:", len(complete_future))
print("Rows available for leakage audit:", len(leakage_audit))

print("\nLeakage-audit columns:")
print(leakage_audit.columns.tolist())

print("\nDuplicate client/content rows after merge:")

duplicate_audit_rows = (
    leakage_audit
    .groupby(["client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
)

print(len(duplicate_audit_rows))

assert len(duplicate_audit_rows) == 0

leakage_audit.head()

Feature-frame rows: 175205
Complete future rows: 74572
Rows available for leakage audit: 73763

Leakage-audit columns:
['client_hash_id', 'content_hash_id', 'imp_prev30', 'clicks_prev30', 'avg_position_prev30', 'content_age_days', 'days_since_last_update', 'imp_next30']

Duplicate client/content rows after merge:
0


,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30,content_age_days,days_since_last_update,imp_next30
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1105.0,2.0,4.447059,396,<NA>,1151.0
1,client_73cda7b4e4f265ea,content_05434271b257bb68,1369.0,6.0,6.990504,396,<NA>,2275.0
2,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2682.0,16.0,3.871738,396,<NA>,6266.0
3,client_73cda7b4e4f265ea,content_712c365258cee05c,5789.0,23.0,4.934013,396,<NA>,8511.0
4,client_73cda7b4e4f265ea,content_8935ed68eca88b01,4640.0,10.0,9.687931,396,<NA>,9670.0


### Create the observed future decline label

I now convert the April outcome into the observed decline label used for the leakage experiment.

A page is considered declining when its April 2026 impressions are less than 80% of its previous 30-day impressions.

Therefore:

`is_declining = 1` if `imp_next30 < 0.8 × imp_prev30`

and `0` otherwise.

This label uses future information deliberately because it represents the outcome I want to predict. It is therefore allowed for evaluation and leakage testing, but it must never be included among the model features.

The label is created only for client-content pairs with a complete 30-day future window.

In [24]:
# Create the observed future decline label.
# This is an outcome variable, NOT a model feature.

leakage_audit["is_declining"] = (
    leakage_audit["imp_next30"]
    < 0.8 * leakage_audit["imp_prev30"]
).astype(int)

print("Observed decline label distribution:")
print(
    leakage_audit["is_declining"]
    .value_counts()
    .sort_index()
)

print("\nObserved decline rate:")
print(
    round(
        leakage_audit["is_declining"].mean() * 100,
        2
    ),
    "%"
)

print("\nLabel rows:", len(leakage_audit))

leakage_audit[
    [
        "imp_prev30",
        "imp_next30",
        "is_declining"
    ]
].head(10)

Observed decline label distribution:
is_declining
0    45950
1    27813
Name: count, dtype: int64

Observed decline rate:
37.71 %

Label rows: 73763


,imp_prev30,imp_next30,is_declining
0,1105.0,1151.0,0
1,1369.0,2275.0,0
2,2682.0,6266.0,0
3,5789.0,8511.0,0
4,4640.0,9670.0,0
5,192.0,331.0,0
6,405.0,235.0,1
7,487.0,354.0,1
8,928.0,522.0,1
9,377.0,252.0,1


### Deliberate label leakage experiment

To demonstrate the danger of leakage, I will deliberately create an unsafe feature from the observed future outcome.

The leaked feature will be the April-to-March impression ratio:

`future_impression_ratio = imp_next30 / imp_prev30`

This feature cannot be known at the March 31 decision point because `imp_next30` comes from April 2026.

It is therefore an intentionally leaked feature.

Because the decline label is itself defined using this future ratio, this feature contains direct information about the label. I expect it to separate declining and non-declining rows extremely well.

This experiment is only a leakage demonstration. The leaked feature will not be included in the approved feature vector.

In [25]:
# Deliberately leaked feature.
# DO NOT use this feature for the real model.

leakage_audit["future_impression_ratio"] = (
    leakage_audit["imp_next30"]
    / leakage_audit["imp_prev30"].replace(0, np.nan)
)

# Remove undefined ratios from this demonstration only.
leak_test = leakage_audit.dropna(
    subset=["future_impression_ratio", "is_declining"]
).copy()

print("Leakage-test rows:", len(leak_test))

print("\nFuture impression ratio summary:")
print(
    leak_test["future_impression_ratio"].describe()
)

print("\nMean ratio by observed label:")
print(
    leak_test
    .groupby("is_declining")["future_impression_ratio"]
    .mean()
)

print("\nMedian ratio by observed label:")
print(
    leak_test
    .groupby("is_declining")["future_impression_ratio"]
    .median()
)

Leakage-test rows: 73763

Future impression ratio summary:
count    73763.000000
mean         8.023117
std        146.828291
min          0.012100
25%          0.656868
50%          0.949903
75%          1.461803
max      13165.000000
Name: future_impression_ratio, dtype: float64

Mean ratio by observed label:
is_declining
0    12.541811
1     0.557760
Name: future_impression_ratio, dtype: float64

Median ratio by observed label:
is_declining
0    1.275460
1    0.584046
Name: future_impression_ratio, dtype: float64


In [26]:
# Reconstruct the decline decision using the deliberately leaked feature.

leak_test["leaked_prediction"] = (
    leak_test["future_impression_ratio"] < 0.8
).astype(int)

leak_match = (
    leak_test["leaked_prediction"]
    == leak_test["is_declining"]
)

print("Leakage reconstruction results:")
print(
    "Matching rows:",
    leak_match.sum()
)

print(
    "Total rows:",
    len(leak_test)
)

print(
    "Match rate:",
    round(leak_match.mean() * 100, 2),
    "%"
)

print("\nConfusion table:")
print(
    pd.crosstab(
        leak_test["is_declining"],
        leak_test["leaked_prediction"],
        rownames=["Observed label"],
        colnames=["Leaked prediction"]
    )
)

Leakage reconstruction results:
Matching rows: 73763
Total rows: 73763
Match rate: 100.0 %

Confusion table:
Leaked prediction      0      1
Observed label                 
0                  45950      0
1                      0  27813


### Remove the deliberately leaked information

The leakage experiment confirmed that `future_impression_ratio` can reproduce the observed decline label with 100% agreement.

This is expected because the leaked feature is calculated from `imp_next30`, which is part of the future outcome used to define the label.

I will now remove the leaked feature and all future outcome fields from the model input.

The final feature vector must contain only the five features approved in ML-04:

- `imp_prev30`
- `clicks_prev30`
- `avg_position_prev30`
- `content_age_days`
- `days_since_last_update`

The observed label is retained separately for evaluation, but it is not included in `X`.

In [27]:
# Fields that are explicitly forbidden from the final model input.
FORBIDDEN_FEATURES = [
    "imp_next30",
    "future_impression_ratio",
    "is_declining",
]

# Construct the final model input from the five approved features only.
X = leakage_audit[FEATURES].copy()

y = leakage_audit["is_declining"].copy()

print("Final model feature columns:")
print(X.columns.tolist())

print("\nTarget column:")
print(y.name)

# Verify that no forbidden field entered X.
leaked_columns_remaining = [
    col for col in FORBIDDEN_FEATURES
    if col in X.columns
]

print("\nForbidden columns remaining in X:")
print(leaked_columns_remaining)

assert leaked_columns_remaining == []

assert X.columns.tolist() == FEATURES

print("\nLeakage removal check passed.")
print("X shape:", X.shape)
print("y shape:", y.shape)

Final model feature columns:
['imp_prev30', 'clicks_prev30', 'avg_position_prev30', 'content_age_days', 'days_since_last_update']

Target column:
is_declining

Forbidden columns remaining in X:
[]

Leakage removal check passed.
X shape: (73763, 5)
y shape: (73763,)


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Fields and information deliberately excluded

I keep the final model input limited to the five features approved in ML-04. The following fields are deliberately excluded because they either contain future information, are derived from the outcome, identify an entity rather than describe its decision-time state, or represent information that is not part of the approved feature contract.

- `imp_next30` — excluded because it is the April 2026 future outcome and would not be known at the March 31 decision moment.

- `is_declining` — excluded from `X` because it is the target label itself. The model should predict this outcome rather than receive it as an input.

- `future_impression_ratio` — excluded because it is deliberately derived from `imp_next30`, so it directly contains future outcome information. The leakage experiment showed that it reproduced the label with 100% agreement.

- `client_hash_id` — retained for grouping, joining, and auditing, but excluded from `X` because it is an identifier rather than a decision-time content feature.

- `content_hash_id` — retained for joining and identifying the page during analysis, but excluded from `X` because it identifies the content item rather than representing one of the approved predictive features.

- Future-window observations — excluded from the feature vector because information from April 2026 belongs to outcome evaluation, not to the March 31 prediction inputs.

- Unapproved content/workflow fields — fields such as publication, deletion, optimization, keyword, provider, or model metadata are not included in the five-feature vector because they were not part of the approved ML-04 feature contract.

The goal is not to claim that every excluded field could never be useful. The narrower claim is that these fields are not part of the current feature vector because they would introduce future information, identifiers, or unapproved information into this particular decision setup.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Explicitly document the fields that must not enter the final model input.

EXCLUDED_FROM_X = [
    "imp_next30",
    "is_declining",
    "future_impression_ratio",
    "client_hash_id",
    "content_hash_id",
]

print("Excluded fields checked:")

for col in EXCLUDED_FROM_X:
    print(f"- {col}")

print("\nChecking that excluded fields are not in X...")

excluded_remaining = [
    col for col in EXCLUDED_FROM_X
    if col in X.columns
]

print("Excluded fields remaining in X:", excluded_remaining)

assert excluded_remaining == []

print("\nAll explicitly excluded fields are absent from X.")

Excluded fields checked:
- imp_next30
- is_declining
- future_impression_ratio
- client_hash_id
- content_hash_id

Checking that excluded fields are not in X...
Excluded fields remaining in X: []

All explicitly excluded fields are absent from X.


### Privacy and identifier check

The warehouse uses pseudonymized identifiers such as `client_hash_id` and `content_hash_id`. I keep these identifiers outside the model matrix so that they can support joins, grouping, and auditing without becoming predictive features.

No client names, URLs, search queries, or other directly identifying fields are included in the final feature vector.

In [29]:
print("Final X columns:")
print(X.columns.tolist())

print("\nIdentifier columns in X:")

identifier_columns = [
    col for col in X.columns
    if "hash_id" in col.lower()
    or col.lower() in ["url", "url_hash_id"]
]

print(identifier_columns)

assert identifier_columns == []

print("\nPrivacy/identifier check passed.")

Final X columns:
['imp_prev30', 'clicks_prev30', 'avg_position_prev30', 'content_age_days', 'days_since_last_update']

Identifier columns in X:
[]

Privacy/identifier check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.